# Exercise A
## Time-varying model

## A note on units
Units in GAMERA are mostly in cgs, with several exceptions that are meaningful in the astrophysical context.

    times are in years, except for particle escape time scales
    lengths are in parsecs
    when retrieving SEDs, for convenience the units are
        erg / s / cm2 vs TeV for radiation flux SEDs
        erg / s vs. TeV for luminosity SEDs (i.e. when the distance to the source is not set)
        erg vs. TeV for particle SEDs

    (differential spectra are again in cgs units)
    gas densities are in number densities.

Also, the units of the spectrum will impact the unit of the calculated radiation spectra. For instance, if the unit of the particle spectrum is only differential in energy, i.e. 1/erg, the output radiation spectra will have the unit of a flux if a source distance is specified (see below) or differential photon count per energy and time if not.

On the other hand, if the input unit is differential also in volume, i.e. 1/erg/cm3, then also the output radiation spectrum will be. Therefore, if no distance is specified in this case, a volume photon emissivity will be calculated, i.e. d3N/dEdVdt.

In [ ]:
import gappa as gp
import numpy as np
import astropy.units as u
from gammapy.maps import MapAxis
import matplotlib.pyplot as plt


### We will consider a leptonic model for which the magnetic field decreases with time and produce a light-curve

First, define the way the magnetic field changes with time. Try to define a function which would result in noticeable changes over the length of the light-curve.

**Tip**: the cooling times plot we made in tutorial 1 is very useful to understand what values of B will result in noticeable changes on a certain timescale!

In [ ]:
def B_field(t,...):
    # define a function here
    return ().to('G')

and plot it to see

In [ ]:
times = np.linspace()*u. # or logspace
B = B_field(times,...) # use your function
plt.loglog(times, B, color='#057775')
plt.xlabel('Time (unit)')
plt.ylabel('B field (G)')
plt.show()

Prepare the B field object the way GAMERA likes

In [ ]:
B_field_lookup = np.array(list(zip(times.to_value('yr'), B.to_value('G'))))


### Define the particle spectrum 

Similar to what we did in the first tutorial, but you can use any shape you want

In [ ]:
def define_injected_spectrum(...):
    
    spectrum =  # define your spectrum and its input parameters
    
    electron_spectrum = np.array(list(zip(energy_electrons.center.to_value('erg'), spectrum)))

    return electron_spectrum

Set up the GAMERA classes we will use

In [ ]:
fp = gp.Particles()
fp.ToggleQuietMode()


Give it the B field lookup we made earlier

In [ ]:
fp.SetBField(B_field_lookup)


Set CMB (or any other) as radiation field

In [ ]:
fp.AddThermalTargetPhotons(T, edens) # define T and edens


Set minimum time if you are using scales less than years (by default GAMERA sets Tmin=1 yr). If you use years, comment this step

In [ ]:
tmin = times[0].to_value('yr')/2
fp.SetTmin(tmin)

Set up the injected spectrum

In [ ]:
power_law_spectrum = define_injected_spectrum()
fp.XXX(power_law_spectrum) # use the right GAMERA method to set up the injected spectrum


Calculate electron spectrum at different time intervals by changing fp.SetAge()

In [ ]:
electron_seds = []
photon_seds = []
time_intervals=np.linspace(times[1], times[-1],6)
for t in time_intervals:
    # set the age and evolve the particle spectrum
    (...)
    
    electron_seds.append(np.array(fp.GetParticleSED()))


and plot the result

In [ ]:
alphas = np.linspace(0.2,1,len(time_intervals))
f = plt.figure(figsize=(10,8))
for s,t,a in list(zip(electron_seds,time_intervals,alphas)):
    plt.loglog(s[:,0],s[:,1],c='#057775',alpha=a,label=str(round(t,1)))
plt.xlabel("E (TeV)")
plt.ylabel("E"+r"$^2$"+"dN/dE (erg)")
plt.grid()
plt.xlim(xmin=1e-6,xmax=2e3) #customize
plt.ylim(ymin=1e36,ymax=1e46) #customize
plt.legend(title="Age(yrs)",ncol=3,fontsize=9) #customize
plt.title("Particle SED") #customize

Now we can make the particles radiate

In [ ]:
distance = 1*u.kpc #customize

fr = gp.Radiation()
fr.ToggleQuietMode()
fr.AddThermalTargetPhotons(T, edens) # same as before
fr.SetDistance(distance.to_value('pc'))
photon_energies = np.logspace(.......)*u.TeV #customize


Loop over the electron SEDs and the times, set the right B and inject the right spectrum

In [ ]:

photon_seds = []

for idx, evolved_electrons in enumerate(electron_seds):
    t = time_intervals[idx]
    B = fp.GetBField(t.to_value('yr'))
    
    fr.XXX(evolved_electrons) # use right GAMERA method to set evolved particles
    fr.XXX(B) # use right GAMERA method to set B field

    # compute the spectrum on the points
    fr.XXX(photon_energies.to_value('erg')) # use right GAMERA method to calculate photon spectrum

    total_sed = np.asarray(fr.GetTotalSED())
    photon_seds.append(total_sed)

and plot the spectrum

In [ ]:

alphas = np.linspace(0.2,1,len(time_intervals))
f = plt.figure(figsize=(10,8))
for s,t,a in list(zip(photon_seds,time_intervals,alphas)):
    plt.loglog(s[:,0],s[:,1],c='#057775',alpha=a,label=str(round(t,1)))
plt.xlabel("E (TeV)")
plt.ylabel("E"+r"$^2$"+"dN/dE (erg cm-2 s-1)")
plt.grid()
#plt.xlim(xmin=1e-6,xmax=2e3)
plt.ylim(ymin=1e-16,ymax=1e-6)
plt.legend(title="Age(days)",ncol=3,fontsize=9)
plt.title("Photon SED")